# 05. Universe Selection & Strategy Construction

## 📋 개요
모델의 예측 결과를 바탕으로 **수익률, 정확도, 위험도를 종합 평가하여 최적의 투자 후보군(Universe)**을 선정합니다.

## ✨ 핵심 전략 및 최근 업데이트
- **v3.9.2**: `리스크점수` 제거 → 5개 위험 지표 개별 컬럼 전환. `filter_statistics.json` 저장 추가.
- **Facade Pattern**: `select_investment_universe()` 단일 함수로 복잡한 평가 로직(수익성, 위험도, 정확도 계산 및 필터링)을 캡슐화하여 실행합니다.
- **사다리꼴 보정 앵커 (v3.9.1)**: `log_return_1d` 타겟 모드 사용 시, 모델의 정확한 성능(RMSE/IC) 평가를 위해 실측 등락률(`target_log_return_1d`)을 앵커(`log_close_ref`)로 주입합니다.
- **비현실적 수익률 필터 (v3.7.2)**: `strategy.max_daily_return` 설정을 통해 비현실적으로 급등하는 거래 시나리오를 자동 제외하고 차선책을 제안합니다.

## 🔄 데이터 흐름
```text
[과거 예측 (test_predictions)] ──┐
[미래 예측 (future_forecasts)] ──┼─→ [ select_investment_universe() ]
[메타 데이터 (dataset.parquet)] ─┘         ├─ 전략 파라미터 적용 (보유 기간, 수익률 상한)
                                           ├─ 평가 (정확도, 수익성, 위험도)
                                           └─ Top-K 후보 선정
                                                  ↓
                                    [ 투자 후보군 (CSV / Parquet) ]
```

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

from src.utils.config import load_config, ProjectPaths
from src.universe.select_universe import select_investment_universe

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로 초기화

In [ ]:
# ==========================================
# 설정 로드 및 경로 초기화 (H2 패턴)
# ==========================================
cfg = load_config()
paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

MODEL_DATE = cfg['universe']['model_date']  # 모델 학습 기준일

print(f"📅 날짜 기준 설정:")
print(f"   - 프로젝트 기준일: {cfg['project']['reference_date']}")
print(f"   - 모델 학습 기준일: {MODEL_DATE}")
print(f"\n📁 경로:")
print(f"   - 과거 예측: {paths.get_predictions_parquet()}")
print(f"   - 미래 예측: {paths.get_forecasts_parquet()}")
print(f"   - Universe 출력: {paths.universe_dir}")

## 2️⃣ 데이터 로드

In [ ]:
print("\n📥 데이터 로드 중...")

# ==========================================
# 1. 과거 예측 결과 (정확도 평가용)
# ==========================================
past_pred_path = paths.training_dir / "test_predictions.parquet"
df_past_pred = pd.read_parquet(past_pred_path)
df_past_pred['date'] = pd.to_datetime(df_past_pred['date'])

print(f"\n[과거 예측]")
print(f"   - 파일: {past_pred_path}")
print(f"   - 행수: {len(df_past_pred):,}")
print(f"   - 기간: {df_past_pred['date'].min()} ~ {df_past_pred['date'].max()}")
print(f"   - 종목 수: {df_past_pred['ticker'].nunique()}")

# ==========================================
# 2. 미래 예측 결과 (수익성 평가용)
# ==========================================
future_pred_path = paths.get_forecasts_parquet()
df_future = pd.read_parquet(future_pred_path)
df_future['date'] = pd.to_datetime(df_future['date'])

print(f"\n[미래 예측]")
print(f"   - 파일: {future_pred_path}")
print(f"   - 행수: {len(df_future):,}")
print(f"   - 기간: {df_future['date'].min()} ~ {df_future['date'].max()}")
print(f"   - 종목 수: {df_future['ticker'].nunique()}")

# ==========================================
# 3. Feature 데이터셋 (리스크 메타 정보)
# ==========================================
dataset_path = paths.get_dataset_parquet()
df_meta = pd.read_parquet(dataset_path)
df_meta['date'] = pd.to_datetime(df_meta['date'])

# 최신 날짜의 메타 정보만 사용
latest_meta_date = df_meta['date'].max()
df_meta_latest = df_meta[df_meta['date'] == latest_meta_date].copy()

print(f"\n[메타 데이터]")
print(f"   - 파일: {dataset_path}")
print(f"   - 기준일: {latest_meta_date}")
print(f"   - 종목 수: {df_meta_latest['ticker'].nunique()}")

print("\n✅ 데이터 로드 완료")

## 3️⃣ Universe 선정 실행 (Facade Pattern)

준비된 데이터와 설정값(`config.yaml`)을 바탕으로 투자 후보를 선정합니다.

### 📐 타겟 모드별 평가 기준 조립
`select_investment_universe` 함수는 타겟 모드에 따라 평가 기준을 동적으로 조정합니다.
- **`log_return_1d` 모드**: 모델 예측값의 역산과 정확도 평가를 위해 기준 가격(`target_log_close`)과 당일 실측 등락률(`target_log_return_1d`)을 `log_close_ref`로 묶어 전달합니다.
- **수익률 상한 필터**: `max_daily_return` 파라미터를 통해 1일 평균 수익률이 상한선을 초과하는 비정상적인 종목은 투자 후보에서 배제합니다.

In [ ]:
print("\n" + "="*65)
print("3️⃣ Universe 선정 실행 (Facade Pattern)")
print("="*65)

# ── 1. 전략 파라미터 로드 ──
strategy_cfg     = cfg.get('strategy', {})
MIN_HOLD_DAYS    = strategy_cfg.get('min_hold_days', 5)
MAX_DAILY_RETURN = strategy_cfg.get('max_daily_return', 0.16)

# ── 2. 타겟 컬럼 조립 ──
train_cfg   = cfg.get('training', {})
target_base = train_cfg.get('target_col_name', 'target_log_close')
horizons    = train_cfg.get('horizons', [1, 2, 3, 4, 5])
TARGET_COLS = [f'{target_base}_h{h}' for h in horizons]

# ── 3. log_close_ref 조립 (log_return_1d 모드 전용 앵커) ──
target_type = train_cfg.get('target_type', 'log_close')

if target_type == 'log_return_1d':
    ref_cols = ['ticker', 'date', 'target_log_close']
    if 'target_log_return_1d' in df_meta.columns:
        ref_cols.append('target_log_return_1d')
    LOG_CLOSE_REF = df_meta[ref_cols].copy()
else:
    LOG_CLOSE_REF = None

print(f"\n[전략 파라미터]")
print(f"   - 최소 보유 기간:  {MIN_HOLD_DAYS}일")
print(f"   - 수익률 상한:     일평균 {MAX_DAILY_RETURN:.1%}")
print(f"   - 평가 타겟 컬럼:  {TARGET_COLS[0]} ~ {TARGET_COLS[-1]}")
print(f"   - 로그 종가 환산:  {'사다리꼴 앵커 적용' if LOG_CLOSE_REF is not None else '미적용 (log_close 모드)'}")

# ── 4. Universe 선정 함수 실행 ──
results = select_investment_universe(
    df_past_predictions=df_past_pred,
    df_future_forecasts=df_future,
    df_meta=df_meta_latest,
    model_date=MODEL_DATE,
    top_k=200,
    min_hold_days=MIN_HOLD_DAYS,
    max_daily_return=MAX_DAILY_RETURN,
    target_columns=TARGET_COLS,
    log_close_ref=LOG_CLOSE_REF,
    verbose=True,
)

# ── 5. 결과 추출 ──
df_accuracy      = results['accuracy']
df_return        = results['returns']
df_risk          = results['risk']
df_full_universe = results['full']
df_candidates    = results['candidates']
filter_stats     = results['filter_stats']

## 4️⃣ 사용자 선택을 위한 상세 리포트 생성

선정된 후보군(`df_candidates`)을 사용자가 읽기 쉽도록 한글 컬럼명으로 정리합니다.
- **수익성 지표**: 예상총수익률(%), 예상일평균수익률, 최적보유기간
- **정확도 지표**: IC(상관계수), 신뢰도(RMSE 역수), RMSE
- **위험 지표**: 변동성, 하방위험, VaR(95%), CVaR(95%), 최대낙폭(MDD)
- **매매 가이드**: 목표 매수일/매도일 및 매수가/매도가

In [ ]:
print("\n" + "="*65)
print("4️⃣ 투자 후보 상세 리포트 생성")
print("="*65)

# 종목명 매핑을 위한 마스터 로드
try:
    master_path = paths.get_ticker_master()
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
    df_candidates['종목명'] = df_candidates['ticker'].map(ticker_name_map)
    print("✅ ticker_master 로드 완료")
except Exception as e:
    print(f"⚠️  ticker_master 로드 실패: {e}")
    df_candidates['종목명'] = df_candidates['ticker']

df_candidates_report = df_candidates.copy()

# 리포트용 한글 컬럼 매핑
df_candidates_report['순위'] = df_candidates_report['return_rank']
df_candidates_report['종목코드'] = df_candidates_report['ticker']

df_candidates_report['예상일평균수익률(로그)'] = df_candidates_report['daily_log_return'].round(6)
df_candidates_report['예상총수익률(%)'] = df_candidates_report['total_return_pct'].round(2)
df_candidates_report['최적보유기간(일)'] = df_candidates_report['hold_days'].astype(int)

df_candidates_report['IC'] = df_candidates_report['ic_mean'].round(4)
df_candidates_report['신뢰도(RMSE역수)'] = df_candidates_report['confidence_rmse'].round(4)
df_candidates_report['RMSE'] = df_candidates_report['rmse'].round(4)

df_candidates_report['변동성'] = df_candidates_report['volatility'].round(4)
df_candidates_report['하방위험'] = df_candidates_report['downside_risk'].round(4)
df_candidates_report['VaR(95%)'] = df_candidates_report['var_95'].round(4)
df_candidates_report['CVaR(95%)'] = df_candidates_report['cvar_95'].round(4)
df_candidates_report['최대낙폭(%)'] = (df_candidates_report['max_drawdown'] * 100).round(2)

df_candidates_report['매수일'] = pd.to_datetime(df_candidates_report['buy_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매도일'] = pd.to_datetime(df_candidates_report['sell_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매수가'] = df_candidates_report['buy_price'].round(0).astype(int)
df_candidates_report['매도가'] = df_candidates_report['sell_price'].round(0).astype(int)
df_candidates_report['유동성점수'] = df_candidates_report['liquidity_score'].round(0).astype(int)

report_cols = [
    '순위', '종목코드', '종목명',
    '예상일평균수익률(로그)', '예상총수익률(%)', '최적보유기간(일)',
    'IC', '신뢰도(RMSE역수)', 'RMSE',
    '변동성', '하방위험', 'VaR(95%)', 'CVaR(95%)', '최대낙폭(%)',
    '매수일', '매도일', '매수가', '매도가',
    '유동성점수'
]

df_report = df_candidates_report[report_cols]

print("\n📊 Top 20 종목 미리보기:")
display_cols_short = [
    '순위', '종목명', '예상총수익률(%)', '최적보유기간(일)',
    'IC', '신뢰도(RMSE역수)', '변동성', '하방위험',
    '매수가', '매도가'
]

display(df_report[display_cols_short].head(20))
print("\n✅ 리포트 생성 완료")

## 5️⃣ 결과 저장

In [ ]:
print("\n" + "="*65)
print("5️⃣ 결과 저장")
print("="*65)

# ==========================================
# 1. 전체 Universe (평가 완료)
# ==========================================
full_universe_path = paths.get_universe_full()
df_full_universe.to_parquet(full_universe_path, index=False)
print(f"\n💾 전체 Universe 저장: {full_universe_path}")
print(f"   - 종목 수: {len(df_full_universe):,}")

# ==========================================
# 2. Top-K 후보 (Parquet)
# ==========================================
candidates_path = paths.get_universe_candidates()
df_candidates.to_parquet(candidates_path, index=False)
print(f"\n💾 투자 후보 저장: {candidates_path}")
print(f"   - 종목 수: {len(df_candidates):,}")

# ==========================================
# 3. 상세 리포트 (CSV, 사람 가독성 우선)
# ==========================================
report_csv_path = paths.get_investment_report_csv()
df_report.to_csv(report_csv_path, index=False, encoding='utf-8-sig')
print(f"\n💾 상세 리포트 저장: {report_csv_path}")
print(f"   - 형식: CSV (Excel 호환)")
print(f"   - 컬럼 수: {len(report_cols)}개")

# ==========================================
# 4. 필터링 통계 (JSON)
# ==========================================
import json as _json
filter_stats_path = paths.get_filter_statistics()
with open(filter_stats_path, 'w', encoding='utf-8') as _f:
    _json.dump(filter_stats, _f, indent=2, ensure_ascii=False)
print(f"\n💾 필터링 통계 저장: {filter_stats_path}")

# ==========================================
# 5. Excel용 요약 시트 (선택)
# ==========================================
try:
    excel_path = paths.get_investment_report_excel()
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: Top 20 요약
        df_report.head(20).to_excel(writer, sheet_name='Top20', index=False)
        
        # Sheet 2: 전체 후보
        df_report.to_excel(writer, sheet_name='전체후보', index=False)
        
        # Sheet 3: 위험별 분류
        df_risk_groups = df_report.copy()
        df_risk_groups['위험등급'] = pd.cut(
            df_risk_groups['변동성'],
            bins=pd.qcut(df_risk_groups['변동성'], q=4, retbins=True)[1],
            labels=['낮음', '보통', '높음', '매우높음']
        )
        
        for risk_level in ['낮음', '보통', '높음', '매우높음']:
            df_level = df_risk_groups[df_risk_groups['위험등급'] == risk_level]
            if len(df_level) > 0:
                df_level.to_excel(writer, sheet_name=f'위험_{risk_level}', index=False)
    
    print(f"\n💾 Excel 리포트 저장: {excel_path}")
    print(f"   - Sheet: Top20, 전체후보, 위험_낮음, 위험_보통 등")
    
except Exception as e:
    print(f"\n⚠️  Excel 저장 실패 (openpyxl 필요): {e}")

print("\n" + "="*65)
print("✅ [Step 5] Universe 선정 완료")
print("="*65)
print(f"\n💡 다음 단계:")
print(f"   1. Excel/CSV 파일 열기: {report_csv_path.name}")
print(f"   2. 수익률, 정확도, 위험을 종합 검토")
print(f"   3. 최종 투자 종목 수동 선택 (권장: 20~30개)")
print(f"   4. 선택한 종목으로 06단계 포트폴리오 최적화 진행")

## 📊 전체 후보 요약 통계 (선택)

In [ ]:
print("\n📊 전체 후보 요약 통계:")

# 수익성 분포
print(f"\n[수익률 분포]")
print(f"   - 10% 이상: {(df_report['예상총수익률(%)'] >= 10).sum()}개")
print(f"   - 5~10%: {((df_report['예상총수익률(%)'] >= 5) & (df_report['예상총수익률(%)'] < 10)).sum()}개")
print(f"   - 0~5%: {((df_report['예상총수익률(%)'] >= 0) & (df_report['예상총수익률(%)'] < 5)).sum()}개")
print(f"   - 음수: {(df_report['예상총수익률(%)'] < 0).sum()}개")

# 보유기간 분포
print(f"\n[보유기간 분포]")
print(f"   - 5일 이하: {(df_report['최적보유기간(일)'] <= 5).sum()}개")
print(f"   - 6~10일: {((df_report['최적보유기간(일)'] > 5) & (df_report['최적보유기간(일)'] <= 10)).sum()}개")
print(f"   - 11~20일: {((df_report['최적보유기간(일)'] > 10) & (df_report['최적보유기간(일)'] <= 20)).sum()}개")
print(f"   - 21일 이상: {(df_report['최적보유기간(일)'] > 20).sum()}개")

# 정확도 분포
print(f"\n[정확도 분포]")
print(f"   - IC 0.7 이상: {(df_report['IC'] >= 0.7).sum()}개")
print(f"   - IC 0.6~0.7: {((df_report['IC'] >= 0.6) & (df_report['IC'] < 0.7)).sum()}개")
print(f"   - IC 0.5~0.6: {((df_report['IC'] >= 0.5) & (df_report['IC'] < 0.6)).sum()}개")
print(f"   - 신뢰도 0.7 이상: {(df_report['신뢰도(RMSE역수)'] >= 0.7).sum()}개")

# 위험도 분포
print(f"\n[위험도 분포]")
_vol_med = df_report['변동성'].median()
_vol_q75 = df_report['변동성'].quantile(0.75)
print(f"   - 변동성 중앙값: {_vol_med:.4f}")
print(f"   - 하방위험 평균: {df_report['하방위험'].mean():.4f}")
print(f"   - VaR(95%) 평균: {df_report['VaR(95%)'].mean():.4f}")
print(f"   - CVaR(95%) 평균: {df_report['CVaR(95%)'].mean():.4f}")
print(f"   - MDD 평균: {df_report['최대낙폭(%)'].mean():.2f}%")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
1. **`universe_candidates.parquet`**: 시스템이 선정한 Top-K 후보군 (데이터 처리용)
2. **`investment_report.csv` / `.xlsx`**: 사용자가 직접 보고 판단할 수 있는 **상세 분석 리포트** ⭐

### 🎯 투자 후보 결정 가이드
생성된 엑셀 리포트를 열고 다음 순서로 검토하는 것을 권장합니다.

1. **상위 20위 검토**: 기대 수익률이 가장 높은 종목들
2. **신뢰도 교차 검증**:
   - `IC`: 최소 60% 이상인가?
   - `신뢰도(RMSE역수)`: 값이 너무 낮지 않은가?
3. **리스크 확인**:
   - `변동성` / `하방위험` / `VaR(95%)` / `CVaR(95%)`: 각 지표를 직접 검토
   - `최대낙폭(%)`: 감당 가능한 수준인가?
4. **매매 계획**:
   - `최적보유기간`과 `매수/매도 목표가` 참고

### 🚀 다음 작업
- **06단계**: 포트폴리오 최적화 (MVO 등)
- **백테스트**: 과거 구간 시뮬레이션